In [1]:
import pandas as pd
wx = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\1. Raw Data\weather_hist.csv")

In [2]:
wx["date"] = pd.to_datetime(wx["date"])
wx = wx.dropna(subset=["EU_HDD", "EU_CDD"]).sort_values("date")
wx = wx.rename(columns={"EU_HDD": "hdd", "EU_CDD": "cdd"})

feb29 = wx["date"].dt.month.eq(2) & wx["date"].dt.day.eq(29)
key = wx["date"].dt.strftime("%m-%d").mask(feb29, "02-28")  # 29 Feb uses the 28 Feb normal
year = wx["date"].dt.year

# 29 February stays out of the 28 February average
hist = wx.loc[~feb29, ["date", "hdd", "cdd"]].copy()
hist["key"] = hist["date"].dt.strftime("%m-%d")
hist["year"] = hist["date"].dt.year

for col, out in [("hdd", "hdd_anom7"), ("cdd", "cdd_anom7")]:
    lookup = hist.set_index(["key", "year"])[col]
    prior = pd.DataFrame({
        lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup)
        for lag in range(1, 11)
    })
    anom = wx[col] - prior.mean(axis=1, skipna=False)  # previous 10 years, same calendar day
    wx[out] = anom.rolling(7, min_periods=7).sum()

In [3]:
wx = wx.drop(columns=["EU_TEMP"])

In [4]:
wx = pd.concat([wx, pd.DataFrame({"date": [wx["date"].max() + pd.Timedelta(days=1)]})], ignore_index=True)
cols = wx.columns.drop("date")
wx[cols] = wx[cols].shift(1)  # gas day D is usable at D+1

In [5]:
wx.to_csv("weather_hist_processed.csv", index=False)